In [1]:
# Importo la librerías necesarias
import requests
import json
import os
import pandas as pd
from datetime import datetime, timedelta
import time

In [2]:
url = "https://nvd.nist.gov/"

# Realizar la petición
response_fecha = requests.get(f'{url}')

# Json to list:
# list_data = response_fecha.json()

# Guardo el total de resultados 
# total_result = list_data['totalResults']
# print(f'La cantidad de resultados {total_result} durante {from_time} - {to_time}' )

In [3]:
response_fecha

<Response [403]>

In [1]:
import time

import sys
import pandas as pd
sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')
from selenium import webdriver
from selenium.webdriver.support.ui import Select
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait 
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC

options = webdriver.ChromeOptions() # Usamos chrome, se podria usar otro.
options.add_argument('--headless') # Chromium sin interfaz grafica
options.add_argument('--no-sandbox') # Seguridad
options.add_argument('--disable-dev-shm-usage') # configuracion de linux
options.add_argument('--user-agent=""Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/74.0.3729.157 Safari/537.36""') # user agent

# driver = webdriver.Chrome()

In [128]:
cve = input("Ingresar un CVE:")
# CVE-2025-20122
# CVE-2024-40591
# CVE-2024-24914
# CVE-2025-20210



Ingresar un CVE: CVE-2025-64155


In [129]:
# Configuramos el web driver
wd = webdriver.Chrome()

# Navegamos la pág NIST con el CVE
url = f"https://nvd.nist.gov/vuln/detail/{cve}"
wd.get(url)

# text_box = wd.find_element(By.ID, "vulnDescriptionTitle")
text_href = wd.find_element(By.XPATH, '//*[@id="vulnTechnicalDetailsDiv"]/table/tbody/tr/td[1]/a')
print(text_href.text)
print(text_href.get_attribute("href"))

CWE-78
http://cwe.mitre.org/data/definitions/78.html


In [130]:
# CWE Common Weakness Enumeration
url_cwe = text_href.get_attribute("href")
wd.get(url_cwe)


try:
    #busco si hay patrones de ataques relacionados
    attack_patterns = wd.find_element(By.ID, "Related_Attack_Patterns")
    
    # En caso de que exista, busco los capec
    href_capec = attack_patterns.find_elements(By.TAG_NAME,"a")
    
    # Guardo los links en una lista
    links_capec = []
    for i in href_capec:
        links_capec.append(i.get_attribute("href"))

except EC.NoSuchElementException:
    print("No contiene patrones de ataque")
    wd.quit()
        
print(links_capec)

["javascript:toggleblocksOC('78_Related_Attack_Patterns');", 'http://capec.mitre.org/data/definitions/108.html', 'http://capec.mitre.org/data/definitions/15.html', 'http://capec.mitre.org/data/definitions/43.html', 'http://capec.mitre.org/data/definitions/6.html', 'http://capec.mitre.org/data/definitions/88.html']


In [131]:
# CAPEC

#busco si hay patrones de ataques relacionados
links_attack = []
for i in range(1,len(links_capec)):
    url_capec = links_capec[i]
    wd.get(url_capec)
    try:
        attack_patterns = wd.find_element(By.ID, "Taxonomy_Mappings")

        # En caso de que exista busco los map_attack
        href_attack = attack_patterns.find_elements(By.TAG_NAME,"a")
        for i in href_attack:
            links_attack.append(i.get_attribute("href"))

    except EC.NoSuchElementException:
        print(f"{url_capec}: No contiene una correlación con mitre")

http://capec.mitre.org/data/definitions/108.html: No contiene una correlación con mitre
http://capec.mitre.org/data/definitions/15.html: No contiene una correlación con mitre
http://capec.mitre.org/data/definitions/6.html: No contiene una correlación con mitre


In [134]:
links_attack
'''
Luego de obtener los links, debo hacer una segunda pasada de verificación.
En caso de tener attack.mitree --> ok
En caso de tener capec.mitre.org/***  -->> volver a buscar el mapping con mitre
En caso contrario descartar.
'''

["javascript:toggleblocksOC('43_Taxonomy Mappings');",
 'https://capec.mitre.org/data/definitions/267.html',
 "javascript:toggleblocksOC('88_Taxonomy Mappings');",
 'http://projects.webappsec.org/OS-Commanding',
 "javascript:toggleblocksOC('267_Taxonomy Mappings');",
 'https://attack.mitre.org/wiki/Technique/T1027']

In [133]:
wd.get('https://capec.mitre.org/data/definitions/267.html')
try:
    attack_patterns = wd.find_element(By.ID, "Taxonomy_Mappings")

    # En caso de que exista busco los map_attack
    href_attack = attack_patterns.find_elements(By.TAG_NAME,"a")
    for i in href_attack:
        links_attack.append(i.get_attribute("href"))

except EC.NoSuchElementException:
    print(f"{url_capec}: No contiene una correlación con mitre")

In [101]:
with open("map_mitre_detections.txt",'a') as f:
    for i in links_attack:
        f.write(f'{i}\n')
    

In [47]:
# Mitre Attack

url_mitre = links_attack[1]
wd.get(url_mitre)

In [127]:
wd.quit()